# Muhawir: train LFM3-1.7B as the support checker (LoRA)

**Before you run:** Settings → Accelerator **GPU T4** (or P100), Internet **on**. Add your data as a Kaggle dataset
named `muhawir-check-data` (the folder made on the server by `scripts/make_check_data.py`: `sft_train.jsonl`,
`sft_dev.jsonl`, `sft_test.jsonl`).

The checker answers one question for each sentence: does it say what its passage says? The last cells measure it on
books it never saw (the test split) and write a GGUF file for Ollama. Run all cells in order (about 1 to 2 hours).

In [ ]:
# 1. Libraries (the versions Liquid's own LoRA recipe uses)
!pip install -q "transformers==5.23.0" "trl==1.18.1" "peft==0.21.3" accelerate datasets gguf sentencepiece

In [ ]:
# 2. Settings: change TRAIN_EXAMPLES to train on more or fewer pairs (more = better and slower)
import glob, json, os, random, torch
BASE = "LiquidAI/LFM3-1.7B"
DATA = (glob.glob("/kaggle/input/*/sft_train.jsonl") or glob.glob("/kaggle/input/*/*/sft_train.jsonl"))[0].rsplit("/", 1)[0]
TRAIN_EXAMPLES, EVAL_EXAMPLES, MAX_TOKENS = 12000, 300, 1024
OUT = "/kaggle/working"
print("data:", DATA, "| GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# 3. Data: each example is the service's own check message and the verdict the checker must give.
# Loss is taken on the verdict only (prompt/completion format).
from datasets import Dataset
def read(name, n=None, seed=1):
    rows = [json.loads(l) for l in open(f"{DATA}/sft_{name}.jsonl", encoding="utf-8")]
    random.Random(seed).shuffle(rows)
    return rows[:n] if n else rows
def as_pc(r):
    m = r["messages"]
    return {"prompt": m[:2], "completion": [m[2]]}
train_rows = read("train", TRAIN_EXAMPLES)
train = Dataset.from_list([as_pc(r) for r in train_rows])
print(len(train), "training pairs; supported:", sum(r["label"] for r in train_rows))

In [ ]:
# 4. Model and LoRA. T4/P100 have no bf16, so float16 is used there
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16 else torch.float16
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE, trust_remote_code=True, dtype=dtype, device_map="auto")
model.gradient_checkpointing_enable()
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")

In [ ]:
# 5. Train (one pass over the data). Watch the loss: it should fall; if it shows nan, set dtype to torch.float32
args = SFTConfig(output_dir=f"{OUT}/run", num_train_epochs=1, per_device_train_batch_size=2,
                 gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
                 max_length=MAX_TOKENS, bf16=bf16, fp16=not bf16, logging_steps=20, save_strategy="no",
                 report_to="none", gradient_checkpointing=True)
trainer = SFTTrainer(model=model, processing_class=tok, train_dataset=train, peft_config=lora, args=args)
trainer.model.print_trainable_parameters()
trainer.train()
trainer.save_model(f"{OUT}/adapter")

In [ ]:
# 6. Measure on books never seen in training: wrong sentences approved (want ~0) and true ones approved (want high)
from collections import Counter
model = trainer.model.eval()
model.config.use_cache = True
def verdict(r):
    enc = tok.apply_chat_template(r["messages"][:2], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    ids = enc["input_ids"].to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=160, do_sample=False)
    text = tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    try:
        start = text.index("{"); data = json.loads(text[start:text.rindex("}") + 1])
        return "approve" if str(data["problem"][0]).strip().lower() == "none" else "reject"
    except Exception:
        return "unusable"
def measure(name):
    rows = read(name, EVAL_EXAMPLES, seed=2)
    got = [(r["type"], r["label"], verdict(r)) for r in rows]
    wrong = [g for g in got if g[1] == 0 and g[2] != "unusable"]
    true = [g for g in got if g[1] == 1 and g[2] != "unusable"]
    print(f"--- {name}: {len(got)} pairs")
    for t, n in sorted(Counter(g[0] for g in got).items()):
        c = Counter(g[2] for g in got if g[0] == t)
        print(f"{t:18} n={n:4}  approve={c['approve']:4}  reject={c['reject']:4}  unusable={c['unusable']:3}")
    print("false approvals:", round(sum(g[2] == 'approve' for g in wrong) / max(1, len(wrong)), 3),
          "| true approved:", round(sum(g[2] == 'approve' for g in true) / max(1, len(true)), 3))
measure("dev"); measure("test")

In [ ]:
# 7. Merge the adapter into the model and write a GGUF file for Ollama (float16; Ollama quantizes it on the server)
merged = trainer.model.merge_and_unload()
merged.save_pretrained(f"{OUT}/merged", safe_serialization=True)
tok.save_pretrained(f"{OUT}/merged")
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/temp/llama.cpp
!pip install -q -r /kaggle/temp/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /kaggle/temp/llama.cpp/convert_hf_to_gguf.py {OUT}/merged --outtype f16 --outfile {OUT}/muhawir-check-f16.gguf
!ls -lh {OUT}/*.gguf

**Download** `muhawir-check-f16.gguf` from the Output panel (and keep the printed scores from step 6).
If step 7 says the architecture is not supported, send the error: the adapter in `adapter/` is still saved.